# GodelReplay: Memory Buffer Sweep
### Does GodelPlugin's contribution grow as replay buffer shrinks?

> **Hypothesis:** The forgetting reduction delta (Replay-only - GodelReplay)
> increases as mem_size decreases, proving GodelPlugin provides complementary
> protection when replay alone is insufficient.

| mem_size | Replay-only forgetting | GodelReplay forgetting | Delta |
|----------|----------------------|----------------------|-------|
| 500 | 0.1500 (known) | 0.1487 (known) | +0.87% |
| 200 | ? | ? | ? |
| 50 | ? | ? | ? |

*Prior result (mem_size=500): GodelReplay marginally better. At smaller buffers, gap should widen.*

**Part of the Two-Layer GodelAI Architecture paper.**
*FLYWHEEL TEAM | MACP v2.3.1 | April 2026*

## 1. Install Dependencies

In [ ]:
import subprocess, sys

def _install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

_install('avalanche-lib', 'torch', 'torchvision')
print('avalanche-lib installed.')


## 2. Load GodelAI Repository

In [ ]:
import subprocess, os, sys

GODELAI_DIR = '/kaggle/working/godelai-repo'

if not os.path.exists(GODELAI_DIR):
    print('Cloning creator35lwb-web/godelai...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/creator35lwb-web/godelai.git', GODELAI_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError('Clone failed: ' + result.stderr)
    print('Cloned successfully.')
else:
    print('godelai-repo already present.')

if GODELAI_DIR not in sys.path:
    sys.path.insert(0, GODELAI_DIR)

from godelai.avalanche_plugin import GodelPlugin
from godelai.strategies.godel_replay import create_godel_replay_strategy
print('GodelPlugin and GodelReplay imported successfully.')


## 3. Configuration

In [ ]:
import torch

def _resolve_device():
    if not torch.cuda.is_available():
        return 'cpu'
    major, minor = torch.cuda.get_device_capability(0)
    if major >= 7:
        return 'cuda'
    print('[Warning] GPU sm_' + str(major) + str(minor) + ' < sm_70 -- using CPU.')
    return 'cpu'

BASE_CONFIG = {
    'n_experiences': 10,
    'seed': 42,
    'train_epochs': 5,
    'train_mb_size': 128,
    'eval_mb_size': 256,
    'lr': 0.001,
    'device': _resolve_device(),
    'ewc_lambda': 400.0,
    'fisher_scaling': 'global_max',
    'propagation_gamma': 2.0,
    't_score_window': 50,
}
MEM_SIZES = [50, 200, 500]

print('Device: ' + BASE_CONFIG['device'])
print('Buffer sizes to test: ' + str(MEM_SIZES))
print('Strategies: replay_only, godel_replay')
print('Total runs: ' + str(len(MEM_SIZES) * 2))
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('GPU: ' + torch.cuda.get_device_name(0) + ' | sm_' + str(cap[0]) + str(cap[1]) + ' | VRAM: ' + str(round(vram,1)) + ' GB')


## 4. Run -- Buffer Size Sweep
> 6 runs total: mem_size=[50, 200, 500] x [replay_only, godel_replay]
> Estimated runtime: 90-150 min on CPU.

In [ ]:
import sys
sys.path.insert(0, GODELAI_DIR)

from experiments.permutedmnist_mem_sweep import run_one, MEM_SIZES, BASE_CONFIG

results = []

for mem_size in MEM_SIZES:
    for strategy in ['replay_only', 'godel_replay']:
        r = run_one(strategy, mem_size, BASE_CONFIG)
        results.append(r)
        forg = '{:.4f}'.format(r['avg_forgetting']) if r['avg_forgetting'] is not None else 'N/A'
        acc  = '{:.4f}'.format(r['final_accuracy'])  if r['final_accuracy']  is not None else 'N/A'
        print('DONE: ' + strategy + ' mem=' + str(mem_size) + ' | forgetting=' + forg + ' acc=' + acc)

print('All runs complete.')


## 5. Results

In [ ]:
print('\n' + '='*72)
print('  MEMORY BUFFER SWEEP -- PermutedMNIST (10 tasks, seed=42)')
print('='*72)
print('  ' + 'mem_size'.ljust(10) + 'Strategy'.ljust(16) +
      'Forgetting'.rjust(12) + 'Accuracy'.rjust(10) + 'Delta'.rjust(12))
print('  ' + '-'*60)

for mem_size in MEM_SIZES:
    r_replay = next((r for r in results if r['strategy'] == 'replay_only' and r['mem_size'] == mem_size), None)
    r_godel  = next((r for r in results if r['strategy'] == 'godel_replay' and r['mem_size'] == mem_size), None)
    if r_replay:
        print('  ' + str(mem_size).ljust(10) + 'replay_only'.ljust(16) +
              '{:.4f}'.format(r_replay['avg_forgetting']).rjust(12) +
              '{:.4f}'.format(r_replay['final_accuracy']).rjust(10) + '  --'.rjust(12))
    if r_godel and r_replay:
        delta = r_replay['avg_forgetting'] - r_godel['avg_forgetting']
        pct   = delta / r_replay['avg_forgetting'] * 100 if r_replay['avg_forgetting'] else 0
        sign  = '+' if pct >= 0 else ''
        print('  ' + str(mem_size).ljust(10) + 'godel_replay'.ljust(16) +
              '{:.4f}'.format(r_godel['avg_forgetting']).rjust(12) +
              '{:.4f}'.format(r_godel['final_accuracy']).rjust(10) +
              (sign + '{:.1f}%'.format(pct)).rjust(12))
        print()

print('='*72)

# Known mem_size=500 results for reference
print('\nReference (from prior run, mem_size=500):')
print('  replay_only   forgetting=0.1500  acc=0.8416')
print('  godel_replay  forgetting=0.1487  acc=0.8418  delta=+0.87%')
print('\nHypothesis: delta should INCREASE as mem_size DECREASES.')


## 6. Interpretation

**What this sweep proves:**

If delta grows as mem_size shrinks, then GodelPlugin and Replay are **complementary**:
- Replay handles distribution shift (what examples to show)
- GodelPlugin handles weight identity (which weights to protect)
- When replay budget is constrained, GodelPlugin fills the gap

**Paper implication:**
> *GodelReplay is most valuable in memory-constrained continual learning settings.
> At large buffer sizes, replay saturates protection and GodelPlugin's contribution
> is marginal. At small buffer sizes, GodelPlugin provides meaningful additional
> forgetting reduction, validating the Two-Layer Architecture complementarity claim.*

---
- [GodelAI GitHub](https://github.com/creator35lwb-web/godelai)
- [GodelAI-Lite Notebook](https://www.kaggle.com/code/creator35lwb/godelai-lite-memory-for-gemma-4)

*creator35lwb | FLYWHEEL TEAM | MACP v2.3.1*